In [2]:
%load_ext autoreload
%autoreload 2

# `Logit` on Orders - Logistic Regression (~1h)

## Select features

🎯 Haydi `wait_time` ve `delay_vs_expected` değişkenlerinin çok `iyi/kötü review`lar üzerindeki etkisini inceleyelim.

👉 `orders` training_set’imizi kullanarak iki adet `multivariate logistic regression` çalıştıracağız:
- `logit_one` → `dim_is_one_star` tahmini için  
- `logit_five` → `dim_is_five_star` tahmini için.

 

In [15]:
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

👉 Dataset’inizi import edin:

In [16]:
from olist.order import Order
orders = Order().get_training_data(with_distance_seller_customer=True)

👉 Kullanmak istediğiniz feature’ları bir listede seçin:

⚠️ Data leakage yaratmadığınızdan emin olun (yani target’tan türetilmiş feature’ları seçmeyin)

💡 `wait_time` ve `delay_vs_expected` değişkenlerinin etkisini anlayabilmek için diğer feature’ların etkisini kontrol etmemiz gerekir, bu yüzden listenize ilgili olabilecek tüm feature’ları dahil edin.

In [17]:
orders.head(5)

,order_id,wait_time,expected_wait_time,delay_vs_expected,order_status,dim_is_five_star,dim_is_one_star,review_score,number_of_items,number_of_sellers,price,freight_value,distance_seller_customer
0,e481f51cbdc54678b7cc49136f2d6af7,8.0,15,-8.0,delivered,0,0,4,1,1,29.99,8.72,18.576110
1,53cdb2fc8bc7dce0b6741e2150273451,13.0,19,-6.0,delivered,0,0,4,1,1,118.70,22.76,851.495069
2,47770eb9100c2d0c44946d9cf07ec65d,9.0,26,-18.0,delivered,1,0,5,1,1,159.90,19.22,514.410666
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0,26,-13.0,delivered,1,0,5,1,1,45.00,27.20,1822.226336
4,ad21c59c0840e6cb83a9ceb5573f8159,2.0,12,-10.0,delivered,1,0,5,1,1,19.90,8.72,29.676625


In [18]:
selected_features = [
    "wait_time",
    "delay_vs_expected",
    "number_of_sellers",
    "distance_seller_customer",
    "price",
    "freight_value"
    ]

🕵🏻 Feature’larınızın `multicollinearity` durumunu `VIF index` kullanarak kontrol edin.

* Çok yüksek olmamalıdır (tercihen < 10), böylece partial regression coefficient’larına ve ilgili `p-values` değerlerine güvenebiliriz.
* Verinizi standardize etmeyi unutmayın!
    * Bir `VIF Analysis`, bir feature’ın diğer feature’lara karşı regresyonunu yaparak hesaplanır...
    * Bu yüzden herhangi bir linear regression çalıştırmadan önce feature’ların `scale etkisini kaldırmak` ve eşit öneme sahip olmalarını sağlamak istersiniz!
    
    
📚 <a href="https://www.statisticshowto.com/variance-inflation-factor/">Statistics How To - Variance Inflation Factor</a>

📚  <a href="https://online.stat.psu.edu/stat462/node/180/">PennState - Detecting Multicollinearity Using Variance Inflation Factors</a>

⚖️ Standardize etme:

In [19]:
orders_standardized = orders.copy()

mu = orders[selected_features].mean()
sigma = orders[selected_features].std()
orders_standardized[selected_features] = (orders[selected_features] - mu) / sigma

orders_standardized

,order_id,wait_time,expected_wait_time,delay_vs_expected,order_status,dim_is_five_star,dim_is_one_star,review_score,number_of_items,number_of_sellers,price,freight_value,distance_seller_customer
0,e481f51cbdc54678b7cc49136f2d6af7,-0.428002,15,0.387964,delivered,0,0,4,1,-0.112544,-0.513802,-0.652038,-0.981488
1,53cdb2fc8bc7dce0b6741e2150273451,0.100519,19,0.585732,delivered,0,0,4,1,-0.112544,-0.086640,0.000467,0.423668
2,47770eb9100c2d0c44946d9cf07ec65d,-0.322297,26,-0.600877,delivered,1,0,5,1,-0.112544,0.111748,-0.164053,-0.145002
3,949d5b44dbf5de918fe9c16f97b45f8a,0.100519,26,-0.106457,delivered,1,0,5,1,-0.112544,-0.441525,0.206815,2.061317
4,ad21c59c0840e6cb83a9ceb5573f8159,-1.062226,12,0.190196,delivered,1,0,5,1,-0.112544,-0.562388,-0.652038,-0.962761
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95875,9c5dedf39a927c1b2549525ed64a053c,-0.428002,18,0.091311,delivered,1,0,5,1,-0.112544,-0.311513,-0.449408,-0.895148
95876,63943bddc261676b46f01ca7ac2f7bd8,1.051855,23,0.981268,delivered,0,0,4,1,-0.112544,0.183977,-0.123156,-0.212974
95877,83c1379a015df1e13d02aae0204711ab,1.263263,30,0.585732,delivered,1,0,5,1,-0.112544,0.333684,1.964490,0.619957
95878,11c177c8e97725db2631073c19f07b62,0.523335,37,-0.897530,delivered,0,0,2,2,-0.112544,1.075186,2.715522,-0.387945


👉 Olası multicollinearity durumlarını analiz etmek için VIF Analysis’inizi çalıştırın:

X_standardized = orders_standardized[selected_features]
X_standardized.head()

In [21]:
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif

df = pd.DataFrame()
df["features"] = X_standardized.columns
df["vif_index"] = [vif(X_standardized.values, i) for i in range(X_standardized.shape[1])]
round(df,2)

,features,vif_index
0,wait_time,2.28
1,delay_vs_expected,1.93
2,number_of_sellers,1.02
3,distance_seller_customer,1.53
4,price,1.21
5,freight_value,1.37


## Logistic Regressions

👉 İki adet `Logistic Regression` modeli fit edin:
- `logit_one` → `dim_is_one_star` tahmini için
- `logit_five` → `dim_is_five_star` tahmini için.

`Logit 1️⃣`

In [22]:
logit_one = smf.logit(formula='dim_is_one_star ~'+ ' + '.join(selected_features), data=orders_standardized).fit()
logit_one.summary()

Optimization terminated successfully.
         Current function value: 0.275872
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:        dim_is_one_star   No. Observations:                95872
Model:                          Logit   Df Residuals:                    95865
Method:                           MLE   Df Model:                            6
Date:                Wed, 28 Oct 2026   Pseudo R-squ.:                  0.1376
Time:                        15:50:37   Log-Likelihood:                -26448.
converged:                       True   LL-Null:                       -30669.
Covariance Type:            nonrobust   LLR p-value:                     0.000
============================================================================================
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                   -2.4826      0.013   -193.275      0.000      -2.508      -2.457
wait_time                    0.6441      0.017     38.067      0.000       0.611       0.677
delay_vs_expected            0.3047      0.017     17.697      0.000       0.271       0.338
number_of_sellers            0.2434      0.008     32.028      0.000       0.228       0.258
distance_seller_customer    -0.1810      0.014    -12.860      0.000      -0.209      -0.153
price                        0.0414      0.011      3.817      0.000       0.020       0.063
freight_value                0.1164      0.011     10.385      0.000       0.094       0.138
============================================================================================
"""

`Logit 5️⃣`

In [23]:
logit_five = smf.logit(formula='dim_is_five_star ~'+ ' + '.join(selected_features), data=orders_standardized).fit()
logit_five.summary()

Optimization terminated successfully.
         Current function value: 0.639445
         Iterations 5


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:       dim_is_five_star   No. Observations:                95872
Model:                          Logit   Df Residuals:                    95865
Method:                           MLE   Df Model:                            6
Date:                Wed, 28 Oct 2026   Pseudo R-squ.:                 0.05419
Time:                        15:50:46   Log-Likelihood:                -61305.
converged:                       True   LL-Null:                       -64817.
Covariance Type:            nonrobust   LLR p-value:                     0.000
============================================================================================
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                    0.3685      0.007     53.979      0.000       0.355       0.382
wait_time                   -0.5585      0.011    -48.861      0.000      -0.581      -0.536
delay_vs_expected           -0.1421      0.010    -14.526      0.000      -0.161      -0.123
number_of_sellers           -0.1787      0.008    -23.346      0.000      -0.194      -0.164
distance_seller_customer     0.1002      0.009     11.638      0.000       0.083       0.117
price                        0.0252      0.008      3.284      0.001       0.010       0.040
freight_value               -0.0645      0.009     -7.497      0.000      -0.081      -0.048
============================================================================================
"""

💡 Şimdi bu iki logistic regression’ın sonuçlarını analiz etme zamanı:

- Partial coefficient’ları kendi kelimelerinizle yorumlayın.
- `p-values` kullanarak istatistiksel anlamlılıklarını kontrol edin.
- Coefficient önemleri açısından `logit_one` ve `logit_five` arasında herhangi bir fark görüyor musunuz?

In [26]:
# Aşağıdaki cümlelerden doğru olanları aşağıdaki listeye kaydedin.

a = "delay_vs_expected influences five_star ratings even more than one_star ratings"
b = "wait_time influences five_star ratings even more than one_star"

your_answer = [a]

🧪 __Kodunu Test Et__

In [27]:
from nbresult import ChallengeResult

result = ChallengeResult('logit',
    answers = your_answer
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/alihan/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/alihan/code/alihantangul/data-logit/tests
plugins: dash-4.4.1, typeguard-4.4.2, anyio-4.8.0
collecting ... collected 1 item

test_logit.py::TestLogit::test_question PASSED                           [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/logit.pickle

git commit -m 'Completed logit step'

git push origin master



<details>
    <summary>- <i>Açıklamalar ve ileri seviye kavramlar</i> -</summary>


> _Diğer tüm şeyler sabitken, `delay factor`, 1-yıldız review alma ihtimalini etkilemesinden bile daha fazla, 5-yıldızdan mahrum kalma ihtimalini artırma eğilimindedir. Muhtemelen bunun sebebi, 1-yıldız review’ların bizzat çok kötü ürünleri hedeflemesi, kötü teslimatları değil._

❗️ Ancak tamamen titiz olmak için, **iki farklı modelin coefficient’larını karşılaştırırken daha dikkatli olmamız gerekir**, çünkü **benzer popülasyonlara dayanmayabilirler**!
    Burada 2 alt popülasyonumuz var: (1-yıldız verenler ve 5-yıldız verenler) ve bunlar doğaları gereği farklı davranış kalıpları sergileyebilirler. 5-yıldız vermeye daha meyilli “mutlu insanlar”ın, “gecikme” veya “fiyat” söz konusu olduğunda, 1-yıldızı “Lucky-Luke gibi ateşleyen” “huysuz insanlara” göre daha az hassas olmaları gayet mümkün...

</details>



🏁 Tebrikler!

💾 `logit.ipynb` notebook’unuzu commit ve push etmeyi unutmayın!